# 游戏销量数据质量审查

从 Raw 生成的审查报表出发，核对全量记录、销量覆盖率和清洗后的主样本数。记录单位是游戏–平台条目，销量单位是百万份。覆盖率的分母包括同一发行年份下有销量和没有销量的 Raw 条目。

先按 [README](../README.md) 从 Raw 生成 `data/processed/` 并读取 [数据字典](../docs/data_dictionary.md)。除最后一格用到画图依赖，其余单元格仅依赖 Python 标准库。需要重绘时，在当前内核对应环境中安装 `matplotlib==3.11.2`。

In [ ]:
import csv
import hashlib
import json
from pathlib import Path

cwd = Path.cwd().resolve()
root = next(
    (directory for directory in (cwd, *cwd.parents)
     if (directory / "src" / "prepare_data.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Start the kernel in the repository or notebooks directory.")

processed = root / "data" / "processed"
manifest_path = processed / "dataset_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Generate data/processed first using the command in README.md.")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
counts = manifest["counts"]

audit_names = (
    "field_missingness",
    "coverage_by_year",
    "coverage_by_platform",
    "coverage_by_period",
    "cleaning_ledger",
)
audit = {}
for name in audit_names:
    relative = f"audit/{name}.csv"
    path = processed / relative
    observed_sha256 = hashlib.sha256(path.read_bytes()).hexdigest()
    if observed_sha256 != manifest["artifacts_sha256"][relative]:
        raise ValueError(f"{relative} does not match the dataset manifest.")
    with path.open(encoding="utf-8", newline="") as handle:
        audit[name] = list(csv.DictReader(handle))

print(f"Dataset version: {manifest['dataset_version']}")
print(f"Raw records: {manifest['raw']['rows']:,}")

## 样本数核对

清洗前候选只要求发行年份落在主区间且总销量有效。主样本还经过记录检查。地区样本要求总销量和各地区的记录完整、可用的正分母，以及地区合计与总量通过一致性检查。优先核对下列计数，再将同一版本交给 EDA 与建模。

In [ ]:
ledger = audit["cleaning_ledger"]
ledger_counts = {row["main_exclusion"]: int(row["records"]) for row in ledger}

assert sum(ledger_counts.values()) == counts["audited"] == manifest["raw"]["rows"]
assert ledger_counts["included"] == counts["main"]
assert all(int(row["input_records"]) == counts["audited"] for row in ledger)

cross_checks = {
    "records": "audited",
    "total_sales_available": "with_total_sales",
    "date_and_total_sales_available": "with_date_and_total_sales",
    "main_candidates": "main_candidates_before_record_checks",
    "main_included": "main",
    "main_regionally_comparable": "main_regional_comparable",
}
for grouping in ("year", "platform", "period"):
    rows = audit[f"coverage_by_{grouping}"]
    for column, manifest_key in cross_checks.items():
        grouped_total = sum(int(row[column]) for row in rows)
        assert grouped_total == counts[manifest_key], (grouping, column, grouped_total)

missingness = {row["field"]: row for row in audit["field_missingness"]}
assert set(missingness) == set(manifest["raw"]["columns"])
observed_totals = counts["audited"] - int(missingness["total_sales"]["audited_missing"])
assert observed_totals == counts["with_total_sales"]

for manifest_key in cross_checks.values():
    print(f"{manifest_key:36s} {counts[manifest_key]:>7,}")

print("\nMain sample ledger")
for category, count in ledger_counts.items():
    print(f"{category:36s} {count:>7,}")

## 发行年份覆盖

上图分母是当年发行的全部 Raw 条目，着色的销量记录数包含已观测到的零销量。下图是销量记录数占当年全部条目的比例。图中仅使用有效发行年份，缺失日期的条目已纳入前文的全量审查。浅蓝背景标出主分析区间，比例按分组内记录数计算，不能解读为市场份额。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, PercentFormatter, StrMethodFormatter

year_rows = audit["coverage_by_year"]
dated_rows = sorted(
    (row for row in year_rows if row["release_year"]),
    key=lambda row: int(row["release_year"]),
)
if not dated_rows:
    raise ValueError("There are no records with a valid release year.")

years = [int(row["release_year"]) for row in dated_rows]
totals = [int(row["records"]) for row in dated_rows]
observed = [int(row["total_sales_available"]) for row in dated_rows]
coverage = [number / total for number, total in zip(observed, totals)]
missing_dates = sum(int(row["records"]) for row in year_rows if not row["release_year"])
first, last = manifest["main_release_years_inclusive"]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "text.color": "#182a45",
    "axes.labelcolor": "#182a45",
    "xtick.color": "#495b70",
    "ytick.color": "#495b70",
})
fig, (top, bottom) = plt.subplots(
    2, 1, sharex=True, figsize=(10.8, 6.7), gridspec_kw={"height_ratios": [1.45, 1]},
)
fig.subplots_adjust(left=0.10, right=0.966, top=0.805, bottom=0.18, hspace=0.22)
for axis in (top, bottom):
    axis.axvspan(first - 0.5, last + 0.5, color="#e6f2fa", linewidth=0, zorder=0)
    axis.set_axisbelow(True)
    axis.grid(axis="y", color="#d5dfe8", linewidth=0.7)
    axis.spines[["top", "right", "left"]].set_visible(False)
    axis.spines["bottom"].set_color("#d5dfe8")
    axis.tick_params(axis="both", length=0, pad=6)

top.bar(years, totals, width=0.76, color="#aebed1", label="All raw records")
top.bar(years, observed, width=0.76, color="#236594", label="Records with global sales")
top.set_ylabel("Records")
top.set_ylim(bottom=0)
top.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
top.legend(
    ncol=2, frameon=False, loc="lower left",
    bbox_to_anchor=(-0.01, 1.015), borderaxespad=0.0, fontsize=10,
)

bottom.bar(years, coverage, width=0.76, color="#236594")
bottom.set_ylabel("Sales coverage")
bottom.set_xlabel("Release year", labelpad=8)
bottom.set_ylim(0, 1.02)
bottom.yaxis.set_major_locator(MultipleLocator(0.25))
bottom.yaxis.set_major_formatter(PercentFormatter(xmax=1))
bottom.xaxis.set_major_locator(MultipleLocator(5))
bottom.set_xlim(min(years) - 0.65, max(years) + 0.65)

fig.text(
    0.10, 0.949, "Global sales coverage by release year",
    fontsize=22, fontweight="bold", va="top",
)
fig.text(
    0.10, 0.897, f"Kaggle Raw game–platform records  |  Main analysis window shaded: {first}–{last}",
    fontsize=10.5, color="#495b70", va="top",
)
fig.text(
    0.10, 0.064,
    f"Coverage = records with global sales / all raw records in the release year. "
    f"{missing_dates:,} records without a valid release year are excluded from the chart.",
    fontsize=8.5, color="#495b70", va="top",
)

figure_path = root / "reports" / "figures" / "coverage_by_release_year.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=160, facecolor="white")
if plt.get_backend().lower() != "agg":
    plt.show()
plt.close(fig)

print(f"Wrote {figure_path.relative_to(root)}")

## 下游交接

- 全局分析同时报告销量观测数和全量条目数，尤其注意少量条目产生的高覆盖率。
- 地区分析使用主样本中 `regional_status == "comparable"` 的记录，并说明地区占比采用的分母。
- 建模使用同一版本的主样本与清单，按 `title_group` 切分。核对字段含义和遗漏情况后，再确定训练内特征处理。